# Code from previous session

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # nvidia (dual 3090s)
print('Using device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
with open('dataset/processed/corpus_clean.txt', 'r') as f:
    text = f.read()

characters = sorted(list(set(text)))
vocab_size = len(characters)
# tokenize - create a mapping between characters to integers
char_to_idx = { ch:i for i,ch in enumerate(characters) }
idx_to_char = { i:ch for i,ch in enumerate(characters) }
encode = lambda xs: [char_to_idx[x] for x in xs]  # encoder: take the string, output the list of integers
decode = lambda xs: ''.join([idx_to_char[x] for x in xs])  # decoder: take the list of integers, output the string

print('Vocab size:', vocab_size)
print('Corpus size:', len(text), 'chars')

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

def get_batch(split, batch_size, context_size):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_size, (batch_size,))
    x = torch.stack([data[i:i+context_size] for i in ix])
    y = torch.stack([data[i+1:i+context_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # lookup table, vocab_size x vocab_size

    def forward(self, idx, targets=None):
        # idx (batch_size, context_size)
        logits = self.token_embedding_table(idx)  # (batch_size, context_size, vocab_size)

        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        else:
            loss = None

        return logits, loss

In [ ]:
def generate(model, context_size, start_idx, number_of_tokens):
    idx = start_idx
    for _ in range(number_of_tokens):
        # crop to last context_size tokens
        idx_cond = idx[:, -context_size:]
        logits, loss = model(idx_cond)
        # apply softmax to get probabilities
        logits = logits[:, -1, :]  # (batch_size, vocab_size)
        probs = F.softmax(logits, dim=1)  # (batch_size, vocab_size)
        idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, t + 1)
    return idx

In [ ]:
@torch.no_grad()
def estimate_loss(model, batch_size, context_size, eval_iters=100):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, context_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


def train(model, steps, batch_size, context_size, report_frequency=1000):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    for step in range(steps):
        # sample a batch of data
        xb, yb = get_batch('train', batch_size, context_size)

        # evaluate the loss
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % report_frequency == 0 or step == steps - 1:
            losses = estimate_loss(model, batch_size, context_size)
            print(f'Step {step}, train loss: {losses["train"]:.4f} val loss: {losses["val"]:.4f}')

In [ ]:
def train_generate_print(model):
    train(model, 5000, 32, 8)

    start_idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    max_tokens = 300
    print(decode(
        generate(model, 8, start_idx=start_idx, number_of_tokens=max_tokens)[0].tolist()
    ))

In [ ]:
m = BigramLanguageModel(vocab_size).to(device)
train_generate_print(m)

# Self-attention mechanism

In [ ]:
# self-attention for a single individual head
torch.manual_seed(1234)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
# B batch, T sequence length, C n_embd

In [ ]:
head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B, T, 16)
q = query(x)  # (B, T, 16)

wei = q @ k.transpose(-2, -1)  # (B, T, 16) @ (B, 16, T) = (B, T, T)

In [ ]:
wei[0]

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei  = F.softmax(wei, dim=-1)

In [ ]:
wei[0]

In [ ]:
out = wei @ x  # (B, T, T) @ (B, T, C) = (B, T, C)

In [ ]:
value = nn.Linear(C, head_size, bias=False)
v = value(x)  # (B, T, 16)

In [ ]:
out = wei @ v  # (B, T, T) @ (B, T, head_size) = (B, T, head_size)

In [ ]:
out

In [ ]:
# Why scale by head_size**-0.5 ?
# Without scaling, wei has high variance -> softmax gets peaky -> gradients vanish
k   = torch.randn(B, T, head_size)
q   = torch.randn(B, T, head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

In [ ]:
q.var()

In [ ]:
wei.var()

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]) * 8, dim=-1)  # gets too peaky, converges to one-hot

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size, n_embd, context_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        # compute attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # causal mask
        wei = F.softmax(wei, dim=-1)  # (B, T, T)
        # weighted aggregation of values
        v   = self.value(x)  # (B, T, head_size)
        out = wei @ v        # (B, T, head_size)
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size, n_embd, context_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, context_size) for _ in range(num_heads)])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return out

In [ ]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head, context_size):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size, n_embd, context_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.sa(x)    # residual connection
        x = x + self.ffwd(x)  # residual connection
        return x

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd=32, context_size=8, n_head=4, n_layer=4):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(context_size, n_embd)
        self.blocks  = nn.Sequential(*[Block(n_embd, n_head=n_head, context_size=context_size) for _ in range(n_layer)])
        self.ln_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                                     # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))       # (T, n_embd)
        x = tok_emb + pos_emb                                                          # (B, T, n_embd)
        x = self.blocks(x)                                                             # (B, T, n_embd)
        logits = self.ln_head(x)                                                       # (B, T, vocab_size)

        if targets is not None:
            B, T, C = logits.shape
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss    = F.cross_entropy(logits, targets)
        else:
            loss = None

        return logits, loss

In [ ]:
m = GPT(vocab_size).to(device)
train_generate_print(m)